# 📈 Batch Inference - Water Quality Predictions

## Purpose
Loads Champion model and runs batch predictions on new water quality samples

## Steps
1. Load Champion model from Unity Catalog
2. Load new water samples for prediction
3. Apply feature engineering pipeline
4. Run batch predictions
5. Write results to inference table with Change Data Feed enabled
6. Generate prediction summary and statistics

## Notes
- Uses Champion model for production predictions
- Writes to inference table for monitoring
- Change Data Feed enabled for Lakehouse Monitoring
- Follows Iris MLOps pattern for production inference

In [ ]:
# 📦 Install required packages
%pip install mlflow scikit-learn pandas numpy

# Restart Python 
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import *

# Initialize
spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")

print("✅ Libraries loaded for batch inference")

In [ ]:
# 🎯 Configuration - Get from Bundle Parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")
schema_name = spark.conf.get("schema_name", "default") 
model_name = spark.conf.get("model_name", "water_quality_model")
mode = spark.conf.get("mode", "batch_inference")

# Construct names
FULL_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"
SOURCE_TABLE = f"{catalog_name}.{schema_name}.water_quality_data"
INFERENCE_TABLE = f"{catalog_name}.{schema_name}.water_quality_inferences"

print(f"🎯 Batch Inference Configuration:")
print(f"   🤖 Model: {FULL_MODEL_NAME}")
print(f"   📊 Source: {SOURCE_TABLE}")
print(f"   📈 Output: {INFERENCE_TABLE}")
print(f"   🎯 Mode: {mode}")

In [ ]:
# 🏆 Load Champion Model
print("🏆 Loading Champion model from Unity Catalog...")

try:
    # Load Champion model
    model_uri = f"models:/{FULL_MODEL_NAME}@Champion"
    champion_model = mlflow.sklearn.load_model(model_uri)
    
    # Get model version info
    client = mlflow.tracking.MlflowClient()
    model_version = client.get_model_version_by_alias(FULL_MODEL_NAME, "Champion")
    
    print(f"✅ Champion model loaded successfully")
    print(f"🔍 Model URI: {model_uri}")
    print(f"📦 Version: {model_version.version}")
    print(f"🏷️ Alias: Champion")
    
except Exception as e:
    print(f"❌ Error loading Champion model: {e}")
    print("💡 Make sure deployment job completed and Champion model exists")
    raise e

In [ ]:
# 📊 Load Data for Inference
print("📊 Loading water quality data for batch inference...")

# Load from source table
source_df = spark.sql(f"SELECT * FROM {SOURCE_TABLE}")

# For demo: simulate new data by taking a sample and adding noise
# In production, this would be actual new water samples
print("🔄 Simulating new water quality samples...")

# Convert to Pandas for processing
source_pandas = source_df.toPandas()

# Create new samples by adding small random variations to existing data
np.random.seed(42)  # For reproducible results
new_samples = source_pandas.sample(n=min(100, len(source_pandas)), random_state=42).copy()

# Add small random noise to simulate new measurements
numeric_cols = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 
               'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

for col in numeric_cols:
    noise = np.random.normal(0, source_pandas[col].std() * 0.05, len(new_samples))
    new_samples[col] = new_samples[col] + noise
    # Ensure values stay positive
    new_samples[col] = np.maximum(new_samples[col], 0.1)

# Remove original sample_id and let it regenerate
new_samples = new_samples.drop(['sample_id', 'ingestion_timestamp', 'data_quality_score'], axis=1, errors='ignore')

print(f"✅ Prepared {len(new_samples)} new samples for inference")
print(f"📊 Sample features: {len(numeric_cols)} water quality parameters")

In [ ]:
# 🔧 Feature Engineering Pipeline
print("🔧 Applying feature engineering pipeline...")

# Apply exact same feature engineering as training
def apply_feature_engineering(df):
    """Apply water quality feature engineering pipeline"""
    df_eng = df.copy()
    
    # Engineered features (MUST match training exactly)
    df_eng['ph_normalized'] = df_eng['ph'] / 14
    df_eng['hardness_ratio'] = df_eng['Hardness'] / (df_eng['Solids'] + 1)
    df_eng['organic_load'] = df_eng['Organic_carbon'] * df_eng['Chloramines']
    
    # Water quality index (composite feature)
    df_eng['water_quality_index'] = (
        df_eng['ph_normalized'] + 
        (1 / (df_eng['Turbidity'] + 1)) + 
        (1 / (df_eng['hardness_ratio'] + 1))
    ) / 3
    
    return df_eng

# Apply feature engineering
new_samples_eng = apply_feature_engineering(new_samples)

# Feature columns for prediction (must match training)
feature_cols = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 
               'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity',
               'ph_normalized', 'hardness_ratio', 'organic_load', 'water_quality_index']

# Prepare features for prediction
X_inference = new_samples_eng[feature_cols]

print(f"✅ Feature engineering applied")
print(f"📊 Features for prediction: {len(feature_cols)}")
print(f"🔍 Shape: {X_inference.shape}")

In [ ]:
# 🚀 Run Batch Predictions
print("🚀 Running batch predictions with Champion model...")

# Make predictions
predictions = champion_model.predict(X_inference)
prediction_probabilities = champion_model.predict_proba(X_inference)

# Add predictions to dataframe
new_samples_eng['predicted_potability'] = predictions
new_samples_eng['prediction_confidence'] = np.max(prediction_probabilities, axis=1)
new_samples_eng['not_potable_probability'] = prediction_probabilities[:, 0]
new_samples_eng['potable_probability'] = prediction_probabilities[:, 1]

# Add metadata
new_samples_eng['model_version'] = model_version.version
new_samples_eng['prediction_timestamp'] = pd.Timestamp.now()
new_samples_eng['model_name'] = FULL_MODEL_NAME
new_samples_eng['inference_id'] = range(len(new_samples_eng))

# Summary statistics
total_predictions = len(predictions)
potable_predictions = (predictions == 1).sum()
not_potable_predictions = (predictions == 0).sum()
avg_confidence = prediction_probabilities.max(axis=1).mean()

print(f"✅ BATCH PREDICTIONS COMPLETED")
print(f"📊 Results Summary:")
print(f"   Total samples: {total_predictions}")
print(f"   ✅ Potable: {potable_predictions} ({potable_predictions/total_predictions*100:.1f}%)")
print(f"   ❌ Not Potable: {not_potable_predictions} ({not_potable_predictions/total_predictions*100:.1f}%)")
print(f"   📈 Avg Confidence: {avg_confidence:.3f}")

In [ ]:
# 💾 Write Results to Inference Table
print("💾 Writing prediction results to inference table...")

# Convert to Spark DataFrame
inference_spark_df = spark.createDataFrame(new_samples_eng)

# Check if inference table exists
table_exists = spark.catalog.tableExists(INFERENCE_TABLE)

if not table_exists:
    print(f"🏗️ Creating new inference table: {INFERENCE_TABLE}")
    
    # Create table with Change Data Feed enabled
    inference_spark_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(INFERENCE_TABLE)
    
    # Enable Change Data Feed for monitoring (required for Lakehouse Monitor)
    spark.sql(f"ALTER TABLE {INFERENCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"✅ Change Data Feed enabled for monitoring")
    
else:
    print(f"📝 Appending to existing inference table: {INFERENCE_TABLE}")
    
    # Append new predictions
    inference_spark_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(INFERENCE_TABLE)

print(f"✅ Predictions written to {INFERENCE_TABLE}")

# Verify the write
total_inferences = spark.sql(f"SELECT COUNT(*) FROM {INFERENCE_TABLE}").collect()[0][0]
print(f"📊 Total inference records in table: {total_inferences:,}")

In [ ]:
# 📊 Inference Results Analysis
print("📊 BATCH INFERENCE ANALYSIS:")
print("=" * 50)

# Recent predictions analysis
recent_stats = spark.sql(f"""
    SELECT 
        COUNT(*) as total_predictions,
        SUM(CASE WHEN predicted_potability = 1 THEN 1 ELSE 0 END) as potable_count,
        SUM(CASE WHEN predicted_potability = 0 THEN 1 ELSE 0 END) as not_potable_count,
        AVG(prediction_confidence) as avg_confidence,
        MIN(prediction_confidence) as min_confidence,
        MAX(prediction_confidence) as max_confidence,
        model_version
    FROM {INFERENCE_TABLE}
    WHERE DATE(prediction_timestamp) = CURRENT_DATE()
    GROUP BY model_version
    ORDER BY model_version DESC
""").collect()

for row in recent_stats:
    print(f"📈 Today's Predictions (Model v{row.model_version}):")
    print(f"   Total: {row.total_predictions}")
    print(f"   ✅ Potable: {row.potable_count} ({row.potable_count/row.total_predictions*100:.1f}%)")
    print(f"   ❌ Not Potable: {row.not_potable_count} ({row.not_potable_count/row.total_predictions*100:.1f}%)")
    print(f"   📊 Confidence: {row.avg_confidence:.3f} (min: {row.min_confidence:.3f}, max: {row.max_confidence:.3f})")

# Confidence distribution
print(f"\n📈 Confidence Distribution:")
confidence_dist = spark.sql(f"""
    SELECT 
        CASE 
            WHEN prediction_confidence >= 0.9 THEN 'High (≥90%)'
            WHEN prediction_confidence >= 0.7 THEN 'Medium (70-90%)'
            ELSE 'Low (<70%)'
        END as confidence_tier,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as percentage
    FROM {INFERENCE_TABLE}
    WHERE DATE(prediction_timestamp) = CURRENT_DATE()
    GROUP BY 1
    ORDER BY count DESC
""").collect()

for row in confidence_dist:
    print(f"   {row.confidence_tier}: {row.count} samples ({row.percentage}%)")

print(f"\n✅ BATCH INFERENCE COMPLETED SUCCESSFULLY!")
print(f"🎯 Results saved to: {INFERENCE_TABLE}")
print(f"📊 Ready for monitoring and drift detection")
print(f"🔄 Next: Run monitoring job to analyze predictions and detect drift")